In [87]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression 
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_absolute_error

In [88]:
df = pd.read_csv('/Users/meetsudra/Documents/GitHub/real-estate/gurgaon_properties_post_feature_selection_2.csv')

In [89]:
df.columns

Index(['property_type', 'sector', 'price', 'built_up_area', 'bedRoom',
       'bathroom', 'balcony', 'agePossession', 'study room', 'servant room',
       'store room', 'pooja room', 'others', 'furnishing_type',
       'luxury_category', 'floor_category'],
      dtype='object')

In [90]:
df.drop(columns=['pooja room', 'store room', 'others'],inplace=True) # we proved that these columns are not at all making ny differnce in the outliers_handling.ipynb file

In [91]:
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,balcony,agePossession,study room,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 86,0.78,1360.0,2,2,2,New Property,0,0,1,Medium,Mid Floor
1,flat,sector 103,1.05,1365.0,2,2,3,Relatively New,0,0,2,Medium,Low Floor
2,flat,sector 70a,0.97,1339.0,3,3,3+,Relatively New,0,0,1,Medium,Low Floor
3,flat,sector 69,2.05,1889.0,4,4,2,Relatively New,0,0,1,High,Low Floor
4,flat,sector 92,1.83,2104.0,4,3,3,New Property,0,1,1,Medium,Mid Floor


In [92]:
df['furnishing_type'].value_counts()

# 1 --> unfurnished
# 2 --> semi
# 0 --> furnished

furnishing_type
1    2471
2    1013
0     194
Name: count, dtype: int64

In [93]:
df['furnishing_type'] = df['furnishing_type'].replace({0:'furnished',2:'semifurnished',1:'unfurnished'})

In [94]:
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,balcony,agePossession,study room,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 86,0.78,1360.0,2,2,2,New Property,0,0,unfurnished,Medium,Mid Floor
1,flat,sector 103,1.05,1365.0,2,2,3,Relatively New,0,0,semifurnished,Medium,Low Floor
2,flat,sector 70a,0.97,1339.0,3,3,3+,Relatively New,0,0,unfurnished,Medium,Low Floor
3,flat,sector 69,2.05,1889.0,4,4,2,Relatively New,0,0,unfurnished,High,Low Floor
4,flat,sector 92,1.83,2104.0,4,3,3,New Property,0,1,unfurnished,Medium,Mid Floor


So we are about to apply all the transformations through a pipeline

In [95]:
X = df.drop(columns=['price'])
y = df['price']

In [96]:
X.columns

Index(['property_type', 'sector', 'built_up_area', 'bedRoom', 'bathroom',
       'balcony', 'agePossession', 'study room', 'servant room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [97]:
X.isnull().sum()

property_type      0
sector             0
built_up_area      0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
study room         0
servant room       0
furnishing_type    0
luxury_category    0
floor_category     0
dtype: int64

In [98]:
# applying the log1p transformation to the target variable 
y_transformed = np.log1p(y)

# Ordinal Encoding -  for tree based models 

In [99]:
columns_to_encode = ['property_type','sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']

In [100]:
# Creating a column transformer for preprocessing 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'study room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ],
    remainder='passthrough'
)

In [101]:
# Creating a pipeline 

pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [102]:
# K-fold cross-validation 
kfold = KFold(n_splits=10, shuffle=True,random_state=42)

In [103]:
X.dtypes

property_type       object
sector              object
built_up_area      float64
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
study room           int64
servant room         int64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object

In [104]:
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [105]:
scores.mean(),scores.std()

(0.7330242118740407, 0.04104543805923992)

In [106]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [107]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [108]:
y_pred = pipeline.predict(X_test)

In [109]:
y_pred = np.expm1(y_pred)

In [110]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9229898372586497

this result is not very good
- We have more models to train and trial so lets rewrite above steps into a function

#### FUNCTION

In [111]:
def scorer(model_name, model):
    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation 
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [112]:

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor


model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}


In [113]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name,model))

In [114]:
model_output

[['linear_reg', 0.7330242118740407, 0.9229898372586497],
 ['svr', 0.7536591607727439, 0.8927988860582411],
 ['ridge', 0.7330279855075734, 0.922979439101745],
 ['LASSO', 0.05059010747080496, 1.5576723433324613],
 ['decision tree', 0.7806554303030286, 0.7102160112959295],
 ['random forest', 0.8877016377454678, 0.4832139759397946],
 ['extra trees', 0.8779660939394995, 0.5282196190679876],
 ['gradient boosting', 0.8767997959677161, 0.5748243735687586],
 ['adaboost', 0.75403748713541, 0.8161184725125787],
 ['mlp', 0.808771240853698, 0.743962614830242],
 ['xgboost', 0.897806519031724, 0.4821880885051644]]

In [115]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [116]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.897807,0.482188
5,random forest,0.887702,0.483214
6,extra trees,0.877966,0.528220
7,gradient boosting,0.876800,0.574824
4,decision tree,0.780655,0.710216
9,mlp,0.808771,0.743963
8,adaboost,0.754037,0.816118
1,svr,0.753659,0.892799
2,ridge,0.733028,0.922979
0,linear_reg,0.733024,0.922990


Best Score :
- xgboost r2: 0.89, mae:0.48

# OneHotEncoding 

- sector column has no order 
- agePossession as well has no order 
- furnishing_type

rest
- floor_category
- luxury_category
- balcony
- property_type has 2 class there fore no need

In [121]:
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category']
onehot_cols = ['sector', 'agePossession', 'furnishing_type']


In [122]:
# Creating a column transformer for preprocessing

preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'study room']),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('oh', OneHotEncoder(drop='first', handle_unknown='ignore'), onehot_cols)
    ],
    remainder='passthrough'
)

In [123]:
# Creating a pipeline 
pipeline = Pipeline([
    ('preprocessorx',preprocessor),
    ('regressor',LinearRegression())
])

In [124]:
X.dtypes

property_type       object
sector              object
built_up_area      float64
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
study room           int64
servant room         int64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object

In [125]:
# K-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [127]:
scores.mean(),scores.std() # ohe se linear model improve kar gaya scores.mean(),scores.std() 

(0.8540684110125975, 0.019593492429722997)

In [128]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [129]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessorx', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ord', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [130]:
y_pred = pipeline.predict(X_test)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [131]:
y_pred = np.expm1(y_pred)

In [132]:
mean_absolute_error(np.expm1(y_test),y_pred) # error also reduced in comparison to ordinal encoding 0.6929288800398369

0.6929288800398369

In [133]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [134]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [135]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

In [46]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [47]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.896003,0.468785
10,xgboost,0.897441,0.498715
1,svr,0.884829,0.500109
5,random forest,0.881747,0.500717
9,mlp,0.883540,0.525813
4,decision tree,0.807246,0.574366
7,gradient boosting,0.859385,0.579541
0,linear_reg,0.854575,0.692929
2,ridge,0.855000,0.694115
8,adaboost,0.722645,0.880716


Best Score : extra trees r2: 0.8953, mae:0.46

altho linear models improved

#### One hot encoding with PCA

In [48]:
# creating a column transformer for preprocessing 
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(), ['bedRoom','bathroom','built_up_area','servant room','study room']),
        ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),['sector','agePossession'])
    ],
    remainder='passthrough'
)

In [49]:

from sklearn.decomposition import PCA
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('pca',PCA(n_components = 0.95)),
    ('regressor', LinearRegression())
])

In [50]:
# K-fold cross-validation 
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [51]:
scores.mean(), scores.std()

(0.05314496327991969, 0.024193226717894385)

mean drastically lowered

In [52]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [53]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [54]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

In [55]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [56]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.770267,0.707355
6,extra trees,0.753178,0.712641
4,decision tree,0.712773,0.792591
10,xgboost,0.591537,0.969389
7,gradient boosting,0.603449,1.026223
1,svr,0.225807,1.393471
8,adaboost,0.283796,1.405043
9,mlp,0.210615,1.442178
3,LASSO,0.050779,1.557586
2,ridge,0.053145,1.562399


we can see performance of every model has dropped

# Target Encoder

U cant target encode before train_test split because model ko test  
- part ka mean values mil sakta he 
- thus we first train test split 
- and then target encode

Altho we are using cross-validations no need to worry

In [57]:
from sklearn.preprocessing import TargetEncoder

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'study room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['agePossession']),
        ('target_enc', TargetEncoder(), ['sector'])  # Using sklearn's built-in TargetEncoder
    ], 
    remainder='passthrough'
)


In [58]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [59]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [60]:
scores.mean(),scores.std()

(0.8395904857247061, 0.02377978319986885)

In [61]:
def scorer(model_name, model):
    output = []
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test), y_pred))
    
    return output

In [62]:
model_dict = {
    'linear_reg': LinearRegression(),
    'svr': SVR(),
    'ridge': Ridge(),
    'LASSO': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor()
}

In [63]:
model_output = []
for model_name, model in model_dict.items():
    try:
        result = scorer(model_name, model)
        model_output.append(result)
    except Exception as e:
        print(f"✗ {model_name} failed: {str(e)}")

In [64]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.904311,0.475659
10,xgboost,0.902219,0.479523
6,extra trees,0.905329,0.482588
7,gradient boosting,0.885718,0.540613
9,mlp,0.854331,0.586170
4,decision tree,0.811028,0.638225
2,ridge,0.839305,0.731381
0,linear_reg,0.839613,0.732954
8,adaboost,0.814770,0.757490
1,svr,0.782718,0.847933


Best results - random forest : r2=0.904, mae = 0.47

- Target encoding works best for tree based models and comparatively better then ordinal encoding for linear models 

Conclusion till now :
- tree based algorithms were performing good especially xgboost and random forest
- One-hot encoding had better results, altho there is not significant difference between target encoding and one-hot encoding in terms of mae

# Hyperparameter tuninig

on random forest 

In [65]:
from sklearn.model_selection import GridSearchCV
para_grid = {
    'regressor__n_estimators':[50,100, 200, 300],
    'regressor__max_depth':[None, 10, 20, 30],
    'regressor__max_samples':[0.1,0.25,0.5,1.0],
    'regressor__max_features':[None,'sqrt']
}

In [66]:
columns_to_encode

['property_type',
 'sector',
 'balcony',
 'agePossession',
 'furnishing_type',
 'luxury_category',
 'floor_category']

In [67]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'study room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['agePossession']),
        ('target_enc', TargetEncoder(), ['sector'])  # Using sklearn's built-in TargetEncoder
    ], 
    remainder='passthrough'
)


In [68]:
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',RandomForestRegressor())
])

In [69]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [70]:
search = GridSearchCV(pipeline, para_grid,cv=kfold, scoring='r2',n_jobs=-1, verbose=4 )

In [71]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits
[CV 1/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.884 total time=   0.2s
[CV 2/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.895 total time=   0.2s
[CV 3/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.885 total time=   0.2s
[CV 4/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.881 total time=   0.2s
[CV 5/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.860 total time=   0.2s
[CV 6/10] END regressor__max_depth=None, regressor__max_features=None, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.887 

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': [None, 'sqrt'], 'regressor__max_samples': [0.1, 0.25, ...], 'regressor__n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,4
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


In [72]:
final_pipe = search.best_estimator_

In [73]:
search.best_params_

{'regressor__max_depth': None,
 'regressor__max_features': None,
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [74]:
search.best_score_

0.9048568076119011

score marginally improved

# Exporting the model

In [143]:
columns_to_encode

['property_type',
 'sector',
 'balcony',
 'agePossession',
 'furnishing_type',
 'luxury_category',
 'floor_category']

In [75]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'study room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [76]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [77]:
X.dtypes

property_type       object
sector              object
built_up_area      float64
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
study room           int64
servant room         int64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object

In [78]:
pipeline.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [79]:
import pickle

with open('dct-viz-tool [real-estate-app]/pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [80]:
with open('dct-viz-tool [real-estate-app]/df.pkl', 'wb') as file:
    pickle.dump(X, file)

both pipeline and dataframes will be needed during website development

In [81]:
X

,property_type,sector,built_up_area,bedRoom,bathroom,balcony,agePossession,study room,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 86,1360.0,2,2,2,New Property,0,0,unfurnished,Medium,Mid Floor
1,flat,sector 103,1365.0,2,2,3,Relatively New,0,0,semifurnished,Medium,Low Floor
2,flat,sector 70a,1339.0,3,3,3+,Relatively New,0,0,unfurnished,Medium,Low Floor
3,flat,sector 69,1889.0,4,4,2,Relatively New,0,0,unfurnished,High,Low Floor
4,flat,sector 92,2104.0,4,3,3,New Property,0,1,unfurnished,Medium,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3673,flat,sector 83,1600.0,3,3,3,Relatively New,0,1,semifurnished,High,Mid Floor
3674,flat,sector 37c,1660.0,3,4,3,Relatively New,0,1,unfurnished,Low,Mid Floor
3675,flat,sector 104,1607.0,3,2,3+,Relatively New,1,0,semifurnished,Medium,High Floor
3676,flat,sector 63,3150.0,4,4,3+,New Property,0,0,semifurnished,Medium,Mid Floor


# Trying out the predictions 

In [136]:
X.columns

Index(['property_type', 'sector', 'built_up_area', 'bedRoom', 'bathroom',
       'balcony', 'agePossession', 'study room', 'servant room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [137]:
X.iloc[0].values

array(['flat', 'sector 86', 1360.0, 2, 2, '2', 'New Property', 0, 0,
       'unfurnished', 'Medium', 'Mid Floor'], dtype=object)

In [ ]:
data = [['flat', 'sector 49', 1, 1, '0', 'New Property', 0.0, 0.0, 0.0, 'furnished', 'High', 'Moderately Old']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'study room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,study room,furnishing_type,luxury_category,floor_category
0,flat,dwarka expressway,1,1,0,New Property,0.0,0.0,0.0,furnished,High,Moderately Old


In [139]:
df['agePossession'].unique()

array(['New Property', 'Relatively New', 'Under Construction',
       'Moderately Old', 'Old Property'], dtype=object)

In [144]:
one_df.dtypes

property_type       object
sector              object
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
built_up_area      float64
servant room       float64
study room         float64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object

In [140]:
np.expm1(pipeline.predict(one_df))


array([0.35366524])

In [128]:
X['sector'].unique().tolist()

['sector 86',
 'sector 103',
 'sector 70a',
 'sector 69',
 'sector 92',
 'sector 56',
 'sector 43',
 'sector 9',
 'sohna road',
 'sector 23',
 'sector 33',
 'sector 12',
 'sector 89',
 'sector 93',
 'sector 37d',
 'sector 7',
 'sector 88a',
 'sector 65',
 'sector 37c',
 'sector 85',
 'sector 68',
 'sector 4',
 'sector 67',
 'sector 67a',
 'sector 83',
 'sector 106',
 'sector 66',
 'sector 24',
 'sector 82',
 'sector 49',
 'sector 108',
 'sector 102',
 'sector 48',
 'sector 81',
 'sector 38',
 'sector 90',
 'sector 25',
 'sector 50',
 'sector 26',
 'sector 21',
 'sector 77',
 'sector 61',
 'sector 105',
 'sector 28',
 'sector 107',
 'sector 73',
 'sector 55',
 'sector 82a',
 'sector 95',
 'sector 112',
 'sector 104',
 'sector 51',
 'sector 76',
 'sector 53',
 'sector 2',
 'sector 91',
 'sector 36a',
 'sector 59',
 'sector 54',
 'sector 31',
 'sector 113',
 'sector 109',
 'sector 58',
 'sector 11',
 'sector 22',
 'sector 71',
 'sector 79',
 'sector 9a',
 'sector 78',
 'sector 3',
 'secto